In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType, DoubleType

catalog_name = 'ecommerce'

In [0]:
df_silver_products = spark.table(f"{catalog_name}.bronze.brz_products")

from_silver_categories_clean = spark.read.table(f"{catalog_name}.silver.slv_categories_clean")

from_silver_brands_clean = spark.read.table(f"{catalog_name}.silver.slv_brands_clean")

In [0]:
row_count, column_count = df_silver_products.count(), len(df_silver_products.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

## 1. product_id

In [0]:
# 1.1 transformation: slv_products -> product_id

df_silver_products = df_silver_products.dropDuplicates(["product_id"])


In [0]:
# 1.2 validation: slv_products -> product_id

df_silver_products.select("product_id").filter(
    F.col("product_id").isNull() |
    (F.length(F.col("product_id")) != 13)) \
    .show()

df_productid_duplicates = df_silver_products.groupBy("product_id") \
    .count() \
    .filter(F.col("count") > 1)


df_silver_products.join(
    df_productid_duplicates,
    on = "product_id",
    how = "inner") \
    .select("product_id").show()


## 2. sku

In [0]:
# 2.1 transformation: slv_products -> sku

In [0]:
# 2.2 validation: slv_products -> sku

df_silver_products.select("sku").filter(F.col("sku").isNull()).show()


## 3. category_code

In [0]:
# 3.1 transformation: slv_products -> category_code

# Transformations done: upper

df_silver_products = df_silver_products.withColumn("category_code", F.upper(F.col("category_code")))

In [0]:
# 3.2 validation: slv_products -> category_code | left-join with df_categories, silver

valid_categories = [
    row["category_code"]
    for row in spark.table(f"{catalog_name}.silver.slv_categories_clean")
    .select("category_code")
    .distinct()
    .collect()
]

df_silver_products.select("category_code").filter(~F.col("category_code").isin(valid_categories)).show()
    

## 4. brand_code

In [0]:
# 4.1 transformation: slv_products -> brand_code

# Transformations done: upper

df_silver_products = df_silver_products.withColumn("brand_code", F.upper(F.col("brand_code")))


In [0]:
# 4.2 validation: slv_products -> brand_code

valid_brands = [
    row["brand_code"]
    for row in spark.table(f"{catalog_name}.silver.slv_brands_clean")
    .select("brand_code")
    .distinct()
    .collect()
]

df_silver_products.select("brand_code").filter(
    ~F.col("brand_code").isin(valid_brands)
).show()

## 5. color

In [0]:
# 5.1 transformation: slv_products -> color

# has nulls, but whatever?

In [0]:
# 5.2 validation: slv_products -> color

df_silver_products.filter(F.col("color").isNull()).show()

## 6. size

In [0]:
# 6.1 transformation: slv_products -> size

size_anomalies = {
  "xs": "XS",
  "s": "S",
  "m": "M",
  "l": "L",
  "xl": "XL",
  "xxl": "XXL",
  "one-size": "One-Size"
}

df_silver_products = df_silver_products.replace(size_anomalies, subset = "size")

In [0]:
# 6.2 validation: slv_products -> size

valid_size = ['S', 'M', 'L', 'XL', 'XXL', 'XS', 'One-Size']

df_silver_products.select("size").filter(
  ~F.col("size").isin(valid_size)
).show()

## 7. material

In [0]:
# 7.1 transformation: slv_products -> material

# no transformations needed

In [0]:
# 7.2 validation: slv_products -> material

valid_material = ['Cotton', 'Rubber', 'Plastic', 'Polyester', 'Steel', 'Leather', 'Wood', 'Aluminium', 'Glass', 'Ceramic']

df_silver_products.select("material").filter(
  ~F.col("material").isin(valid_material)) \
  .show()

## 8. weight_grams

In [0]:
# 8.1 transformation: slv_products -> weight_grams

df_silver_products = df_silver_products.withColumn(
    "weight_grams", 
    F.regexp_replace(F.col("weight_grams"), "g", "").cast(DoubleType())
)

In [0]:
# 8.2 validation: slv_products -> weight_grams

df_silver_products.select("weight_grams") \
    .filter(F.col("weight_grams").isNull() |
        (F.col("weight_grams") < 0)) \
    .show()

## 9. length_cm

In [0]:
# 9.1 transformation: slv_products -> length_cm

#changed type: string -> double

df_silver_products = df_silver_products.withColumn(
  "length_cm", 
  F.round(F.col("length_cm").cast(DoubleType()))
  )
                                                   

In [0]:
# 9.2 validation: slv_products -> length_cm

df_silver_products.select("length_cm").filter(
  F.col("length_cm").isNull() |
  (F.col("length_cm") < 0)
).show()

## 10. width_cm

In [0]:
# 10.1 transformation: slv_products -> width_cm
df_silver_products = df_silver_products.withColumn(
    "width_cm",
    F.round(F.col("width_cm").cast(DoubleType()), 2)
)

In [0]:
# 10.2 validation: slv_products -> width_cm

df_silver_products.select("width_cm").filter(
    F.col("width_cm").isNull() |
    (F.col("width_cm") < 0)
).show()

## 11. height_cm

In [0]:
# 11.1 transformation: slv_products -> height_cm

df_silver_products = df_silver_products.withColumn(
    "height_cm",
    F.round(F.col("height_cm").cast(DoubleType()), 2)
)


In [0]:
# 11.2 validation: slv_products -> height_cm

df_silver_products.select("height_cm").filter(
  F.col("height_cm").isNull() |
  (F.col("height_cm") < 0)
).show()

## 12. rating

In [0]:
# 12.1 transformation: slv_products -> rating

df_silver_products = df_silver_products.withColumn(
    "rating", F.round(F.col("rating").cast("double"), 2)
)


In [0]:
# 12.2 validation: slv_products -> rating

df_silver_products.select("rating").filter(
  F.col("rating").isNull() |
  (F.col("rating") < 0)
).show()

## 13. rating_count

In [0]:
# 13.1 transformation: slv_products -> rating_count

df_silver_products = df_silver_products.withColumn(
  "rating_count",
  F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
    .otherwise(F.lit(0))
    )
  


In [0]:
# 13.2 validation: slv_products -> rating_count

df_silver_products.select("rating_count").filter(
  F.col("rating_count") < 0
).show()

In [0]:
display(df_silver_products)

## quarantine vs clean => silver_products

In [0]:
df_silver_products_clean = df_silver_products.filter(
  F.col("product_id").isNotNull() & (F.col("product_id") != "") & (F.length(F.col("product_id")) == 13) &
  F.col("sku").isNotNull() & (F.col("sku") != "") &
  F.col("brand_code").isNotNull() & (F.col("brand_code") != "") & (F.col("brand_code").isin(valid_brands)) &
  F.col("category_code").isNotNull() & (F.col("category_code") != "") & (F.col("category_code").isin(valid_categories)) &
  F.col("size").isNotNull() & (F.col("size") != "") & (F.col("size").isin(valid_size)) &
  F.col("material").isNotNull() & (F.col("material") != "") & (F.col("material").isin(valid_material)) &
  F.col("weight_grams").isNotNull() & (F.col("weight_grams") >= 0 ) &
  F.col("width_cm").isNotNull() & (F.col("width_cm") >= 0 ) &
  F.col("height_cm").isNotNull() & (F.col("height_cm") >= 0 ) &
  F.col("rating").isNotNull() & (F.col("rating") >= 0 ) &
  F.col("rating_count").isNotNull() & (F.col("rating_count") >= 0)
)

df_silver_products_quarantine = df_silver_products.filter(
  F.col("product_id").isNull() | (F.col("product_id") == "") | (F.length(F.col("product_id")) != 13) |
  F.col("sku").isNull() | (F.col("sku") == "") |
  F.col("brand_code").isNull() | (F.col("brand_code") == "") | (~F.col("brand_code").isin(valid_brands)) |
  F.col("category_code").isNull() | (F.col("category_code") == "") | (~F.col("category_code").isin(valid_categories)) |
  F.col("size").isNull() | (F.col("size") == "") | (~F.col("size").isin(valid_size)) |
  F.col("material").isNull() | (F.col("material") == "") | (~F.col("material").isin(valid_material)) |
  F.col("weight_grams").isNull() | (F.col("weight_grams") < 0 ) |
  F.col("width_cm").isNull() | (F.col("width_cm") < 0 ) |
  F.col("height_cm").isNull() | (F.col("height_cm") < 0 ) |
  F.col("rating").isNull() | (F.col("rating") < 0 ) |
  F.col("rating_count").isNull() | (F.col("rating_count") < 0)) \
    .withColumn("rejection_reason",
      F.when(F.col("product_id").isNull() | (F.col("product_id") == "") | (F.length(F.col("product_id")) != 13), "null/empty product_id or wrong length")
      .when(F.col("sku").isNull() | (F.col("sku") == ""), "null/empty sku")
      .when(F.col("brand_code").isNull() | (F.col("brand_code") == "") | (~F.col("brand_code").isin(valid_brands)), "null/empty brand_code or invalid brand_code")
      .when(F.col("category_code").isNull() | (F.col("category_code") == "") | (~F.col("category_code").isin(valid_categories)), "null/empty category_code or invalid category_code")
      .when(F.col("size").isNull() | (F.col("size") == "") | (~F.col("size").isin(valid_size)), "null/empty size or invalid size")
      .when(F.col("material").isNull() | (F.col("material") == "") | (~F.col("material").isin(valid_material)), "null/empty material or invalid material")
      .when(F.col("weight_grams").isNull() | (F.col("weight_grams") < 0 ), "null/invalid weight_grams")
      .when(F.col("width_cm").isNull() | (F.col("width_cm") < 0 ), "null/invalid width_cm")
      .when(F.col("height_cm").isNull() | (F.col("height_cm") < 0 ), "null/invalid height_cm")
      .when(F.col("rating").isNull() | (F.col("rating") < 0), "null/invalid rating")
      .when(F.col("rating_count").isNull() | (F.col("rating_count") < 0), "null/invalid rating_count")
      .otherwise("invalid product")
    )


In [0]:
print(f"Clean Count: {df_silver_products_clean.count()}")
print(f"Quarantine Count: {df_silver_products_quarantine.count()}")
print(f"Original Count: {df_silver_products.count()}")

df_silver_products_quarantine.show()

In [0]:
df_silver_products_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_products_clean")

df_silver_products_quarantine.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_products_quarantine")